# SAR Oil Spill Segmentation - Gradio App (Google Colab)
#### Выполнила: Фирсова Ольга

Интерфейс для запуска обученной модели UNet (encoder=resnet34) на загруженных SAR `.tif` файлах (каналы VV, VH)

**Перед запуском:**
1. Загрузите на свой Google Drive: `best_model.pth`, `shared_config.json`, и опционально `background.jpg`
2. Выполните ячейки по порядку, на втором шаге Colab запросит доступ к Google Drive
3. Укажите правильные пути к файлам в ячейке с путями (PATH ниже)

Ссылка на `best_model.pth`: https://www.kaggle.com/datasets/daryanikitina/unet-resnet34-oil-spill-sar-segmentation

In [1]:
!pip install -q gradio segmentation-models-pytorch rasterio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.4 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import json
from pathlib import Path

import gradio as gr
import numpy as np
import rasterio
import segmentation_models_pytorch as smp
import torch
import torch.nn as nn

CHECKPOINT_PATH = Path("/content/drive/MyDrive/gradio-files/best_model.pth")
SHARED_CONFIG_PATH = Path("/content/drive/MyDrive/gradio-files/shared_config.json")
BACKGROUND_PATH = Path("/content/drive/MyDrive/gradio-files/background.jpg")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cuda


In [4]:
with open(SHARED_CONFIG_PATH) as f:
    shared_config = json.load(f)

PATCH_SIZE = shared_config["PATCH_SIZE"]
VV_MEAN = shared_config["norm_stats"]["VV_mean"]
VV_STD = shared_config["norm_stats"]["VV_std"]
VH_MEAN = shared_config["norm_stats"]["VH_mean"]
VH_STD = shared_config["norm_stats"]["VH_std"]

DEFAULT_PIXEL_SIZE_M = 10.0
THRESHOLD = 0.5

print("Loaded config:", shared_config["norm_stats"])

Loaded config: {'VV_mean': -31.315275, 'VV_std': 7.87874, 'VH_mean': -19.719892, 'VH_std': 6.16873}


## Модель

In [5]:
def build_model():
    model = smp.Unet(
        encoder_name="resnet34",
        encoder_weights=None,
        in_channels=2,
        classes=1,
    )
    return model.to(DEVICE)


print(f"Loading model on {DEVICE}...")
model = build_model()
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint["model_state"])
model.eval()
print(f"Loaded checkpoint from epoch {checkpoint['epoch']}.")

Loading model on cuda...
Loaded checkpoint from epoch 11.


## Предобработка и инференс

In [6]:
def load_sar_tif(file_path):
    with rasterio.open(file_path) as src:
        if src.count < 2:
            raise ValueError(f"Expected a 2-band SAR file, got {src.count} band(s).")
        vv = src.read(1).astype(np.float32)
        vh = src.read(2).astype(np.float32)
    return np.stack([vv, vh], axis=-1)


def normalize(img):
    img = img.copy()
    img[:, :, 0] = (img[:, :, 0] - VV_MEAN) / VV_STD
    img[:, :, 1] = (img[:, :, 1] - VH_MEAN) / VH_STD
    return img


def pad_to_patch_size(img, patch_size):
    h, w = img.shape[:2]
    pad_h = (patch_size - h % patch_size) % patch_size
    pad_w = (patch_size - w % patch_size) % patch_size
    if pad_h or pad_w:
        img = np.pad(img, ((0, pad_h), (0, pad_w), (0, 0)), mode="constant")
    return img, (h, w)


@torch.no_grad()
def run_inference(img_norm, patch_size):
    padded, (orig_h, orig_w) = pad_to_patch_size(img_norm, patch_size)
    full_h, full_w = padded.shape[:2]
    prob_canvas = np.zeros((full_h, full_w), dtype=np.float32)

    for r in range(0, full_h, patch_size):
        for c in range(0, full_w, patch_size):
            patch = padded[r:r + patch_size, c:c + patch_size, :]
            tensor = torch.from_numpy(patch).permute(2, 0, 1).unsqueeze(0).float().to(DEVICE)
            prob = torch.sigmoid(model(tensor)).squeeze().cpu().numpy()
            prob_canvas[r:r + patch_size, c:c + patch_size] = prob

    return prob_canvas[:orig_h, :orig_w]


def colorize_mask(mask_bin):
    h, w = mask_bin.shape
    overlay = np.zeros((h, w, 4), dtype=np.uint8)
    overlay[mask_bin > 0] = [255, 0, 0, 140]
    return overlay


def vv_to_grayscale(vv_raw):
    """Adaptive per-image percentile stretch — robust to varying dB ranges
    across different uploaded files."""
    lo, hi = np.percentile(vv_raw, [2, 98])
    stretched = np.clip((vv_raw - lo) / (hi - lo + 1e-6), 0, 1)
    return (stretched * 255).astype(np.uint8)

In [7]:
raw = load_sar_tif("/content/drive/MyDrive/gradio-files/00006.tif")
print("VV min/max/mean:", raw[:,:,0].min(), raw[:,:,0].max(), raw[:,:,0].mean())
print("VH min/max/mean:", raw[:,:,1].min(), raw[:,:,1].max(), raw[:,:,1].mean())

VV min/max/mean: -54.781693 1.6561728 -38.00397
VH min/max/mean: -48.00087 9.050487 -21.935303


In [8]:
import rasterio
with rasterio.open("/content/drive/MyDrive/gradio-files/00006.tif") as src:
    print("Band count:", src.count)
    print("Dtype:", src.dtypes)
    print("Width/Height:", src.width, src.height)

Band count: 2
Dtype: ('float32', 'float32')
Width/Height: 2048 2048


## Основная функция (вызывается кнопкой)

In [9]:
def predict(file_obj, pixel_size_m):
    threshold = THRESHOLD

    if file_obj is None:
        return None, None, "Загрузите файл .tif с каналами VV и VH."
    if pixel_size_m is None or pixel_size_m <= 0:
        return None, None, "Размер пикселя должен быть положительным числом (в метрах)."

    try:
        raw = load_sar_tif(file_obj)
    except Exception as e:
        return None, None, f"Ошибка чтения файла: {e}"

    img_norm = normalize(raw)
    prob = run_inference(img_norm, PATCH_SIZE)
    mask_bin = (prob > threshold).astype(np.uint8)

    vv_gray = vv_to_grayscale(raw[:, :, 0])
    vh_gray = vv_to_grayscale(raw[:, :, 1])
    base_rgb = np.concatenate(
        [np.stack([vv_gray] * 3, axis=-1), np.stack([vh_gray] * 3, axis=-1)],
        axis=1,
    )

    pixel_area_km2 = (pixel_size_m / 1000.0) ** 2
    oil_area_km2 = int(mask_bin.sum()) * pixel_area_km2
    total_area_km2 = mask_bin.size * pixel_area_km2

    info = (
        f"Общая площадь снимка: {total_area_km2:.4f} км²\n"
        f"Обнаруженная площадь нефти: {oil_area_km2:.4f} км²\n"
        f"Размер пикселя: {pixel_size_m:.1f} × {pixel_size_m:.1f} м\n"
        f"Размер снимка: {raw.shape[1]}×{raw.shape[0]} px"
    )
    return base_rgb, (mask_bin * 255).astype(np.uint8), info

## Стили (CSS) и тексты интерфейса (HTML)

In [10]:
CSS = """
/* ═══════════════════════════════════════════════════════════════════════════
   SAR Oil Spill Segmentation — dark blue theme
   ═══════════════════════════════════════════════════════════════════════════ */

/* ── Background image ── */
.gradio-container {
    background-image: url('/gradio_api/file=/content/drive/MyDrive/gradio-files/background.jpg');
    background-size: cover;
    background-position: center;
    background-attachment: fixed;
    min-height: 100vh;
    font-family: 'Segoe UI', 'Arial', sans-serif;
}

/* ── Translucent overlay so content is readable ── */
.gradio-container::before {
    content: '';
    position: fixed;
    inset: 0;
    background: rgba(4, 12, 36, 0.72);
    z-index: 0;
    pointer-events: none;
}

/* ── All direct children sit above the overlay ── */
.gradio-container > * {
    position: relative;
    z-index: 1;
}

/* ── Header block ── */
.header-block {
    padding: 36px 40px 24px;
    border-bottom: 1px solid rgba(100, 160, 255, 0.25);
    margin-bottom: 8px;
}

.header-block .header-tagline {
    font-size: 0.78rem;
    font-weight: 600;
    letter-spacing: 1.5px;
    text-transform: uppercase;
    color: rgba(180, 205, 240, 0.85);
    margin-bottom: 10px;
}

.header-block h1 {
    font-size: 2rem;
    font-weight: 700;
    color: #e8f0ff;
    letter-spacing: 0.5px;
    margin: 0 0 8px;
    text-shadow: 0 2px 12px rgba(0, 80, 255, 0.4);
}
.header-block p {
    font-size: 0.97rem;
    color: #a8c0e8;
    margin: 0 0 10px;
    line-height: 1.6;
}
.header-block .note {
    display: inline-block;
    font-size: 0.85rem;
    color: #7ab0f0;
    background: rgba(30, 60, 120, 0.5);
    border-left: 3px solid #3a7bd5;
    padding: 6px 14px;
    border-radius: 0 6px 6px 0;
}

/* ── Panel cards ── */
.panel-card {
    background: rgba(8, 20, 55, 0.75);
    border: 1px solid rgba(80, 130, 220, 0.3);
    border-radius: 14px;
    padding: 24px;
    backdrop-filter: blur(8px);
    box-shadow: 0 4px 24px rgba(0, 30, 100, 0.4);
}

/* ── Section labels ── */
.section-label {
    font-size: 1.2rem;
    font-weight: 700;
    letter-spacing: 1.2px;
    text-transform: uppercase;
    color: #7ab8f5;
    margin-bottom: 16px;
    padding: 10px 18px;
    display: inline-block;
    background: rgba(20, 50, 120, 0.55);
    border: 1px solid rgba(90, 150, 230, 0.45);
    border-radius: 10px;
    box-shadow: 0 2px 10px rgba(0, 30, 100, 0.35);
}

/* ── Gradio component label overrides ── */
label span,
.label-wrap span {
    color: #c8dcf8 !important;
    font-size: 0.88rem !important;
    font-weight: 500 !important;
}

/* ── File upload box ── */
.upload-btn,
[data-testid="upload-btn"],
.file-preview {
    background: rgba(15, 35, 80, 0.8) !important;
    border: 1.5px dashed rgba(80, 140, 220, 0.5) !important;
    border-radius: 10px !important;
    color: #a8c8f0 !important;
}
.upload-btn:hover {
    border-color: rgba(100, 170, 255, 0.8) !important;
    background: rgba(20, 50, 110, 0.9) !important;
}

/* ── Number input ── */
input[type=number],
input[type=text] {
    background: rgba(15, 35, 80, 0.9) !important;
    border: 1px solid rgba(80, 140, 220, 0.4) !important;
    border-radius: 8px !important;
    color: #e0ecff !important;
    padding: 10px 14px !important;
    font-size: 1rem !important;
}
input[type=number]:focus,
input[type=text]:focus {
    border-color: #4a90d9 !important;
    box-shadow: 0 0 0 2px rgba(74, 144, 217, 0.25) !important;
    outline: none !important;
}

/* ── Run button ── */
button.primary,
button[variant="primary"] {
    background: linear-gradient(135deg, #1a4fa8 0%, #0d2d6e 100%) !important;
    border: 1px solid rgba(100, 160, 255, 0.4) !important;
    border-radius: 10px !important;
    color: #e8f0ff !important;
    font-size: 1rem !important;
    font-weight: 600 !important;
    padding: 13px 28px !important;
    letter-spacing: 0.4px;
    transition: all 0.2s ease !important;
    box-shadow: 0 3px 14px rgba(26, 79, 168, 0.5) !important;
    width: 100% !important;
}
button.primary:hover,
button[variant="primary"]:hover {
    background: linear-gradient(135deg, #2460c8 0%, #1440a0 100%) !important;
    box-shadow: 0 4px 20px rgba(36, 96, 200, 0.7) !important;
    transform: translateY(-1px) !important;
}

/* ── Image output panels ── */
.image-container,
[data-testid="image"] {
    background: rgba(8, 20, 55, 0.8) !important;
    border: 1px solid rgba(80, 130, 220, 0.25) !important;
    border-radius: 10px !important;
}

/* ── Textbox (summary) ── */
textarea {
    background: rgba(8, 20, 55, 0.85) !important;
    border: 1px solid rgba(80, 130, 220, 0.3) !important;
    border-radius: 10px !important;
    color: #c8dcf8 !important;
    font-size: 0.92rem !important;
    line-height: 1.7 !important;
    padding: 12px 16px !important;
}

/* ── Output image labels ── */
.output-image .label-wrap span {
    color: #7ab0f0 !important;
}

/* ── Footer ── */
.footer-note {
    text-align: center;
    font-size: 1.0rem;
    color: rgba(140, 170, 220, 0.6);
    padding: 20px 0 12px;
    border-top: 1px solid rgba(80, 130, 220, 0.15);
    margin-top: 16px;
}


/* Apply panel card style directly to Gradio columns (fixes empty-box bug) */
.gradio-container .gr-column,
.gradio-container .form {
    background: rgba(8, 20, 55, 0.75) !important;
    border: 1px solid rgba(80, 130, 220, 0.3) !important;
    border-radius: 14px !important;
    padding: 24px !important;
    backdrop-filter: blur(8px) !important;
    box-shadow: 0 4px 24px rgba(0, 30, 100, 0.4) !important;
}

"""

In [11]:
HTML_TEMPLATE = """
<!-- ═══════════════════════════════════════════════════════════════════════
     SAR Oil Spill Segmentation — HTML fragments
     Loaded at runtime by gradio_app.py
     ═══════════════════════════════════════════════════════════════════════ -->

<!-- HEADER -->
<div class="header-block">
    <div class="header-tagline">Oil spill segmentation · Sentinel-1 SAR · Deep Learning Pipeline</div>
    <h1>🛰 Сегментация разливов нефти на SAR-снимках</h1>
</div>

<!-- INPUTS LABEL -->
<div class="section-label">📂 Входные данные</div>

<!-- RESULTS LABEL -->
<div class="section-label">🌍 Результаты</div>

<!-- FOOTER -->
<div class="footer-note"></div>

"""

def _html(tag):
    marker = f"<!-- {tag} -->"
    start = HTML_TEMPLATE.find(marker)
    if start == -1:
        return ""
    start += len(marker)
    next_comment = HTML_TEMPLATE.find("<!--", start)
    block = HTML_TEMPLATE[start: next_comment if next_comment != -1 else len(HTML_TEMPLATE)]
    return block.strip()

## Интерфейс Gradio

In [12]:
with gr.Blocks(title="SAR Oil Spill Segmentation", css=CSS) as demo:

    gr.HTML(_html("HEADER"))

    with gr.Row(equal_height=False):

        with gr.Column(scale=1):
            gr.HTML(_html("INPUTS LABEL"))
            file_input = gr.File(label="SAR .tif файл (VV + VH)", file_types=[".tif", ".tiff"])
            pixel_size_input = gr.Number(
                value=DEFAULT_PIXEL_SIZE_M, precision=1, minimum=0.1,
                label="Размер пикселя (м/сторону) - Sentinel-1: 10 м",
            )
            run_button = gr.Button("Запустить сегментацию", variant="primary")

        with gr.Column(scale=2):
            gr.HTML(_html("RESULTS LABEL"))
            with gr.Row():
                output_original = gr.Image(label="Исходный снимок (VV слева | VH справа)", elem_classes=["output-image"], height=400)
                output_mask = gr.Image(label="Бинарная маска (белый = нефть)", elem_classes=["output-image"], height=400)
            output_info = gr.Textbox(label="Сводка", lines=5, interactive=False)

    gr.HTML(_html("FOOTER"))

    run_button.click(
        fn=predict,
        inputs=[file_input, pixel_size_input],
        outputs=[output_original, output_mask, output_info],
    )

/tmp/ipykernel_789/2969393897.py:1: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(title="SAR Oil Spill Segmentation", css=CSS) as demo:


## Запуск

В Colab используйте `share=True` для получения публичной ссылки (Colab не даёт прямого доступа к localhost, как и Kaggle).

In [13]:
demo.launch(share=True, allowed_paths=["/content/drive/MyDrive/gradio-files/"])

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://637e5f2f4a063702d7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
